## Loading and Evaluating a Foundation Model

In [143]:
# Loading the model
from transformers import AutoModelForSequenceClassification
import torch
model = AutoModelForSequenceClassification.from_pretrained("gpt2", num_labels=2)
print(model)

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 2614.80it/s, Materializing param=transformer.wte.weight]             
GPT2ForSequenceClassification LOAD REPORT from: gpt2
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


GPT2ForSequenceClassification(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (score): Linear(in_features=768, out_features=2, bias=False)
)


In [144]:
# Creating mapping for labels and ids
model.config.id2label = {
    0: "ENTAILMENT",
    1: "NOT_ENTAILMENT"
}

In [145]:
model.config.label2id = {
    "ENTAILMENT": 0,
    "NOT_ENTAILMENT": 1
}

# Evaluating the model

In [146]:
# Loading the dataset
from datasets import load_dataset

splits = ["train", "validation"]
ds = {split: ds for split, ds in zip(splits, load_dataset("glue", "rte", split=splits))}
print(ds["validation"][0])


{'sentence1': 'Dana Reeve, the widow of the actor Christopher Reeve, has died of lung cancer at age 44, according to the Christopher Reeve Foundation.', 'sentence2': 'Christopher Reeve had an accident.', 'label': 1, 'idx': 0}


In [147]:
# Loading tokenizer
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

def tokenize_function(x):
    text = [
        f"Text: {s1}\nHypothesis: {s2}"
        for s1, s2 in zip(x['sentence1'], x['sentence2'])
    ]
    return tokenizer(
        text,
        truncation=True
    )

# Tokenizing the data
tokenized_ds = {}
for split in splits:
    tokenized_ds[split] = ds[split].map(tokenize_function, batched=True)
tokenized_ds["train"] = tokenized_ds["train"].rename_column("label", "labels")
tokenized_ds["validation"] = tokenized_ds["validation"].rename_column("label", "labels")
print(tokenized_ds["train"][0]['input_ids'])

[8206, 25, 1400, 18944, 286, 5674, 25034, 4062, 287, 3908, 6430, 13, 198, 49926, 313, 8497, 25, 18944, 286, 5674, 25034, 4062, 287, 3908, 13]


In [148]:
tokenized_ds

{'train': Dataset({
     features: ['sentence1', 'sentence2', 'labels', 'idx', 'input_ids', 'attention_mask'],
     num_rows: 2490
 }),
 'validation': Dataset({
     features: ['sentence1', 'sentence2', 'labels', 'idx', 'input_ids', 'attention_mask'],
     num_rows: 277
 })}

In [149]:
import torch
from transformers import DataCollatorWithPadding
from torch.utils.data import DataLoader

model.config.pad_token_id = tokenizer.eos_token_id

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Remove non-tensor columns so DataCollatorWithPadding can batch cleanly
columns_to_remove = ["sentence1", "sentence2", "idx"]
test_ds = tokenized_ds["validation"].remove_columns(columns_to_remove)

data_loader = DataLoader(
    test_ds,
    batch_size=8,
    collate_fn=data_collator
)

all_logits=[]

with torch.no_grad():
    for batch in data_loader:
        outputs = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"]
        )
        all_logits.append(outputs.logits)



In [150]:
print(all_logits)

[tensor([[ 1.8660, -8.9043],
        [ 2.2593, -9.8866],
        [ 2.0773, -9.6557],
        [ 2.5551, -9.3880],
        [ 1.1794, -6.4302],
        [ 1.7999, -7.6725],
        [ 1.6473, -8.5463],
        [ 1.1055, -6.7703]]), tensor([[ 1.4534, -8.0839],
        [ 1.5071, -7.7872],
        [ 1.9065, -8.4110],
        [ 0.6883, -5.4389],
        [ 2.1469, -8.4801],
        [ 1.6848, -8.8107],
        [ 2.1864, -9.4981],
        [ 1.6632, -8.5414]]), tensor([[ 1.0230, -6.3881],
        [ 1.4049, -7.9175],
        [ 1.2296, -6.8162],
        [ 1.7572, -8.6454],
        [ 1.2337, -6.5300],
        [ 1.5404, -8.4267],
        [ 1.4768, -7.2111],
        [ 1.8713, -8.6262]]), tensor([[ 1.7768, -9.1769],
        [ 1.1445, -5.1569],
        [ 1.2885, -7.0096],
        [ 1.2702, -7.7671],
        [ 1.9320, -8.2318],
        [ 1.7295, -8.8549],
        [ 2.2867, -8.2475],
        [ 1.8754, -8.7578]]), tensor([[ 1.7200, -9.0058],
        [ 1.9484, -9.7145],
        [ 2.0433, -7.0533],
        [ 1

In [151]:
preds = [logit.argmax(dim=1) for logit in all_logits]
print(preds)

[tensor([0, 0, 0, 0, 0, 0, 0, 0]), tensor([0, 0, 0, 0, 0, 0, 0, 0]), tensor([0, 0, 0, 0, 0, 0, 0, 0]), tensor([0, 0, 0, 0, 0, 0, 0, 0]), tensor([0, 0, 0, 0, 0, 0, 0, 0]), tensor([0, 0, 0, 0, 0, 0, 0, 0]), tensor([0, 0, 0, 0, 0, 0, 0, 0]), tensor([0, 0, 0, 0, 0, 0, 0, 0]), tensor([0, 0, 0, 0, 0, 0, 0, 0]), tensor([0, 0, 0, 0, 0, 0, 0, 0]), tensor([0, 0, 0, 0, 0, 0, 0, 0]), tensor([0, 0, 0, 0, 0, 0, 0, 0]), tensor([0, 0, 0, 0, 0, 0, 0, 0]), tensor([0, 0, 0, 0, 0, 0, 0, 0]), tensor([0, 0, 0, 0, 0, 0, 0, 0]), tensor([0, 0, 0, 0, 0, 0, 0, 0]), tensor([0, 0, 0, 0, 0, 0, 0, 0]), tensor([0, 0, 0, 0, 0, 0, 0, 0]), tensor([0, 0, 0, 0, 0, 0, 0, 0]), tensor([0, 0, 0, 0, 0, 0, 0, 0]), tensor([0, 0, 0, 0, 0, 0, 0, 0]), tensor([0, 0, 0, 0, 0, 0, 0, 0]), tensor([0, 0, 0, 0, 0, 0, 0, 0]), tensor([0, 0, 0, 0, 0, 0, 0, 0]), tensor([0, 0, 0, 0, 0, 0, 0, 0]), tensor([0, 0, 0, 0, 0, 0, 0, 0]), tensor([0, 0, 0, 0, 0, 0, 0, 0]), tensor([0, 0, 0, 0, 0, 0, 0, 0]), tensor([0, 0, 0, 0, 0, 0, 0, 0]), tensor([0, 0,

In [152]:
predicted_labels = [
    model.config.id2label[predicted_class_id] 
    for batch in preds
    for predicted_class_id in batch.tolist()
    ]
print(predicted_labels)

['ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTA

In [153]:
count_correct = 0
for index, item in enumerate(ds['validation']):
    if item['label'] == model.config.label2id[predicted_labels[index]]:
        count_correct+=1
print(f"Accuracy: {count_correct/len(predicted_labels)}")

Accuracy: 0.5270758122743683


In [154]:
# Creating a PEFT config
# LoraConfig documentation: https://huggingface.co/docs/peft/main/en/conceptual_guides/lora
from peft import LoraConfig
config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["c_attn", "c_proj"], # typically attention or MLP blocks
    lora_dropout=0.1,
    bias="lora_only", # could be 'none', 'all', or 'lora_only'
    modules_to_save=["score"],
    #use_rslora=True, # try with and without
)

In [155]:
# Combining the two
from peft import get_peft_model
lora_model = get_peft_model(model, config)

/Users/olha/vscode101/Udacity/hf_env/lib/python3.11/site-packages/peft/tuners/lora/layer.py:2285: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


In [156]:
for name, param in lora_model.named_parameters():
    if "score" in name:
        param.requires_grad = True

In [157]:
# Checking Trainable Parameters of a PEFT Model
lora_model.print_trainable_parameters()

trainable params: 1,671,168 || all params: 126,064,896 || trainable%: 1.3256


In [158]:
import numpy as np
from transformers import DataCollatorWithPadding, Trainer, TrainingArguments
from sklearn.metrics import f1_score

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return {
        "accuracy": (predictions==labels).mean(),
        "f1": f1_score(labels, predictions, average="macro")
    }

columns_to_remove = ["sentence1", "sentence2", "idx"]

for split in splits:
    tokenized_ds[split] = tokenized_ds[split].remove_columns(columns_to_remove)

trainer = Trainer(
    model=lora_model,
    args=TrainingArguments(
        output_dir="./data/isEntailment",
        learning_rate=5e-4,
        lr_scheduler_type="cosine",
        warmup_steps=0.1,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=8,
        eval_strategy='epoch',
        save_strategy='epoch',
        num_train_epochs=15,
        weight_decay=0.01,
        load_best_model_at_end=True,
        max_grad_norm=1.0,
    ),
    train_dataset=tokenized_ds['train'],
    eval_dataset=tokenized_ds['validation'],
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)
trainer.train()

/Users/olha/vscode101/Udacity/hf_env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,0.709534,0.523466,0.350320
2,No log,0.644753,0.642599,0.636459
3,No log,0.656024,0.620939,0.618072
4,0.924227,0.685589,0.606498,0.550197
5,0.924227,0.650938,0.646209,0.645169
6,0.924227,0.707724,0.624549,0.621942
7,0.574077,0.722628,0.657040,0.649736
8,0.574077,0.753879,0.667870,0.663088
9,0.574077,0.737489,0.678700,0.674887
10,0.381572,0.881169,0.711191,0.710735


/Users/olha/vscode101/Udacity/hf_env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/Users/olha/vscode101/Udacity/hf_env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/Users/olha/vscode101/Udacity/hf_env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/Users/olha/vscode101/Udacity/hf_env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/Users/olha/vscode10

TrainOutput(global_step=2340, training_loss=0.4617300490028838, metrics={'train_runtime': 1623.0728, 'train_samples_per_second': 23.012, 'train_steps_per_second': 1.442, 'total_flos': 3230895051491328.0, 'train_loss': 0.4617300490028838, 'epoch': 15.0})

In [159]:
# Evaluate
trainer.evaluate()

/Users/olha/vscode101/Udacity/hf_env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


{'eval_loss': 0.6475734710693359,
 'eval_accuracy': 0.6462093862815884,
 'eval_f1': 0.6382848310414668,
 'eval_runtime': 4.0072,
 'eval_samples_per_second': 69.125,
 'eval_steps_per_second': 8.734,
 'epoch': 15.0}

In [160]:
# Saving a Trained PEFT Model
lora_model.save_pretrained("gpt-lora")

In [161]:
# Loading a Saved PEFT Model
from peft import AutoPeftModelForSequenceClassification
lora_model = AutoPeftModelForSequenceClassification. from_pretrained("gpt-lora")

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 979.50it/s, Materializing param=transformer.wte.weight]             
GPT2ForSequenceClassification LOAD REPORT from: gpt2
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [164]:
lora_model.config.pad_token_id = tokenizer.eos_token_id

In [165]:
# Evaluating the model
trainer = Trainer(
    model=lora_model,
    eval_dataset=tokenized_ds['validation'],
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)

In [166]:
trainer.evaluate()

/Users/olha/vscode101/Udacity/hf_env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


{'eval_loss': 0.6467854976654053,
 'eval_model_preparation_time': 0.0064,
 'eval_accuracy': 0.631768953068592,
 'eval_f1': 0.6300282840980516,
 'eval_runtime': 4.5416,
 'eval_samples_per_second': 60.991,
 'eval_steps_per_second': 7.706}